# Naive Bayes

### Overview

- [Setup and Data Loading](#Setup-and-Data-Loading)
- [Data Understanding](#Data-Understanding)
- [Encoding the Features](#Encoding-the-Features)
- [The Majority Baseline](#The-Majority-Baseline)
- [A First Naive Bayes Model](#A-First-Naive-Bayes-Model)
- [Tuning Naive Bayes](#Tuning-Naive-Bayes)
- [Final Evaluation and the Imbalance Effect](#Final-Evaluation-and-the-Imbalance-Effect)
- [Saving the Results](#Saving-the-Results)

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report

## Setup and Data Loading

The train/test split and the leakage-safe publisher_tier are already created in Phase B.

In [2]:
SCOPE = "scope_2000_2018"
DATA_DIR = "data/processed/" + SCOPE

X_train = pd.read_csv(DATA_DIR + "/X_train.csv")
X_test  = pd.read_csv(DATA_DIR + "/X_test.csv")
y_train = pd.read_csv(DATA_DIR + "/y_train.csv")["Hit"]
y_test  = pd.read_csv(DATA_DIR + "/y_test.csv")["Hit"]

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
X_train.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/scope_2000_2018/X_train.csv'

## Data Understanding

The model only gets the three pre-release features from Phase B.

For Naive Bayes the class prior is important, so I check the class balance first.

In [ ]:
print('Hit  (1):', len(y_train[y_train == 1]) / len(y_train))
print('Flop (0):', len(y_train[y_train == 0]) / len(y_train))

print()
print('Training labels:')
print(y_train.value_counts().sort_index())

## Encoding the Features

CategoricalNB cannot directly use strings. I use OrdinalEncoder to convert categories to integer IDs.

For this model the numbers are only category IDs. They are not distances.

In [ ]:
X_train_cat = X_train.astype(str)
X_test_cat = X_test.astype(str)

for col in X_train_cat.columns:
    unseen = sorted(set(X_test_cat[col]) - set(X_train_cat[col]))
    print(col, 'unseen test categories:', unseen)

enc = OrdinalEncoder()
X_train_enc = enc.fit_transform(X_train_cat)
X_test_enc = enc.transform(X_test_cat)

print('Encoded training shape:', X_train_enc.shape)
X_train_enc[:5]

## The Majority Baseline

A simple baseline always predicts the majority class, which is Flop.

In [ ]:
baseline_pred = np.zeros(len(y_test), dtype=int)

print('Baseline accuracy:', accuracy_score(y_test, baseline_pred))
print('Baseline F1      :', f1_score(y_test, baseline_pred, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, baseline_pred))

## A First Naive Bayes Model

First I train a basic categorical Naive Bayes model with the default Laplace smoothing.

In [ ]:
nb_first = CategoricalNB()
nb_first.fit(X_train_enc, y_train)

pred = nb_first.predict(X_test_enc)
proba = nb_first.predict_proba(X_test_enc)[:, 1]

print('Test accuracy:', accuracy_score(y_test, pred))
print('Test F1      :', f1_score(y_test, pred))
print('Test ROC-AUC :', roc_auc_score(y_test, proba))
print('Confusion matrix:')
print(confusion_matrix(y_test, pred))

## Tuning Naive Bayes

In [ ]:
param_grid = {'alpha': [0.1, 0.5, 1.0, 2.0, 5.0]}

nb_model = CategoricalNB()
nb_cv = GridSearchCV(nb_model, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
nb_cv.fit(X_train_enc, y_train)

print('Best CV ROC-AUC:', nb_cv.best_score_)
print('Best parameters:', nb_cv.best_params_)

## Final Evaluation and the Imbalance Effect

The final comparison uses the same tuned procedure twice:

1. train on the original imbalanced training set
2. train on a 50/50 down-sampled training set

The test set is not changed.

In [ ]:
def evaluate(model, X_te, y_te, model_name, resample):
    proba = model.predict_proba(X_te)[:, 1]
    pred = model.predict(X_te)

    roc = roc_auc_score(y_te, proba)
    f1 = f1_score(y_te, pred)
    acc = accuracy_score(y_te, pred)

    print('--- %s (resample=%s) ---' % (model_name, resample))
    print('ROC-AUC : %.3f' % roc)
    print('F1      : %.3f' % f1)
    print('Accuracy: %.3f' % acc)
    print('Confusion matrix [rows = true 0/1, cols = predicted 0/1]:')
    print(confusion_matrix(y_te, pred))
    print(classification_report(y_te, pred, zero_division=0))

    return {'model': model_name, 'resample': resample,
            'roc_auc': roc, 'f1': f1, 'accuracy': acc}


def fit_nb(X_tr, y_tr):
    gs = GridSearchCV(CategoricalNB(), param_grid,
                      scoring='roc_auc', cv=5, n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)
    print('best parameters:', gs.best_params_)
    return gs.best_estimator_

### Run 1 — no resampling

In [ ]:
nb_none = fit_nb(X_train_enc, y_train)
row_none = evaluate(nb_none, X_test_enc, y_test, 'naive_bayes', 'none')

### Run 2 — 50/50 down-sampling

In [ ]:
train = pd.DataFrame(X_train_enc)
train['Hit'] = y_train.values

hits = train[train['Hit'] == 1]
flops = train[train['Hit'] == 0]

flops_down = flops.sample(n=len(hits), random_state=0)
train_bal = pd.concat([hits, flops_down]).sample(frac=1, random_state=0)

X_train_bal = train_bal.drop(columns='Hit')
y_train_bal = train_bal['Hit']

print('Balanced training set size:', len(y_train_bal))
print('Hit  (1):', len(y_train_bal[y_train_bal == 1]))
print('Flop (0):', len(y_train_bal[y_train_bal == 0]))

In [ ]:
nb_down = fit_nb(X_train_bal, y_train_bal)
row_down = evaluate(nb_down, X_test_enc, y_test, 'naive_bayes', 'downsample')

## Saving the Results

The two runs are saved with the same columns as the other Phase C model notebooks.

In [ ]:
os.makedirs('results', exist_ok=True)

results = pd.DataFrame([row_none, row_down])
results = results[['model', 'resample', 'roc_auc', 'f1', 'accuracy']]

out_path = 'results/naive_bayes.csv'
results.to_csv(out_path, index=False)
print('Saved', out_path)
results